In [97]:
import pandas as pd
import random
import numpy as np
from datetime import datetime
import h5py
from camera import Camera

In [98]:
def make_samples(cam, n_bin, drop_rate, add_rate, label, n_samples, seed, hr_to_idx):
    samples = np.empty((n_samples, 2))
    mask = cam.data["hr"] == label
    gpx = cam.data[mask]["px"]
    gpy = cam.data[mask]["py"]
    cam.data.drop(index=gpx.index, inplace=True)

    max_r = np.sqrt((cam.resolution[0]**2) + (cam.resolution[1]**2)) / 2
    bins = max_r * np.sqrt(np.linspace(0, 1, n_bin + 1))

    drop_n_stars = int(cam.data["hr"].size * drop_rate)
    add_n_stars = int(cam.data["hr"].size * add_rate)

    rng = np.random.default_rng(seed)

    for n in range(n_samples):
        coords = np.array(cam.data[["px", "py"]])
        coords[:, 0] += rng.normal(0, 5, coords[:, 0].shape)
        coords[:, 1] += rng.normal(0, 5, coords[:, 1].shape)
        coords[:, 0] = np.clip(coords[:, 0], 0, cam.resolution[1])
        coords[:, 1] = np.clip(coords[:, 1], 0, cam.resolution[0])
        
        coords[:, 0] -= gpx.to_numpy()
        coords[:, 1] -= gpy.to_numpy()
        
        distances = np.linalg.norm(coords, axis=1)
        histo = np.histogram(distances, bins)
        
        result = histo[0] + rng.multinomial(add_n_stars, [(1/(histo[0].size))] * (histo[0].size))
        result -= rng.multinomial(drop_n_stars, [(1/(histo[0].size))] * (histo[0].size))
        result = np.clip(result, 0, None)


        with h5py.File("data.hdf5", "a") as f:
            mapped_label = hr_to_idx[label]
            if str(mapped_label) not in f:
                grp = f.create_group(str(mapped_label))
            else:
                grp = f[str(mapped_label)]
            
            grp.create_dataset(f"sample_{n}", data=result)

    f.close()

In [99]:
seed = 1
random.seed(seed)

data_path = "./hygdata_v42.csv"
data = pd.read_csv(data_path)
data.drop_duplicates(subset="hr", inplace=True)
mask = data["hr"].notna()
cam = Camera(30, [1024,1024], data[mask])


In [100]:
unique_hr_ids = sorted(set(hr for hr in data["hr"]))
hr_to_idx = {hr_id: i for i, hr_id in enumerate(unique_hr_ids)}
idx_to_hr = {i: hr_id for hr_id, i in hr_to_idx.items()}
type(hr_to_idx[1.0])

int

In [101]:
s = 0
for index, row in data[mask].iterrows():
    label = row["hr"]
    print(f"Making data for hr:{label} index: {index}!")
    cam.point("hr", label)
    cam.prep()
    make_samples(cam, 25, 0.1, 0.1, label, 360, seed, hr_to_idx)
    cam.reset_data(data[mask])
    s += 1
print("Done!")
print(f"{s} Stars accounted for!")

Making data for hr:9077.0 index: 25!
Making data for hr:9078.0 index: 34!
Making data for hr:9079.0 index: 43!
Making data for hr:9080.0 index: 63!
Making data for hr:9081.0 index: 88!
Making data for hr:9083.0 index: 106!
Making data for hr:9082.0 index: 107!
Making data for hr:9084.0 index: 122!
Making data for hr:9085.0 index: 124!
Making data for hr:9086.0 index: 137!
Making data for hr:9087.0 index: 145!
Making data for hr:9089.0 index: 154!
Making data for hr:9088.0 index: 171!
Making data for hr:9091.0 index: 183!
Making data for hr:9092.0 index: 186!
Making data for hr:9093.0 index: 194!
Making data for hr:9094.0 index: 208!
Making data for hr:9095.0 index: 238!
Making data for hr:9096.0 index: 253!
Making data for hr:9097.0 index: 274!
Making data for hr:9098.0 index: 300!
Making data for hr:9099.0 index: 302!
Making data for hr:9100.0 index: 330!
Making data for hr:9101.0 index: 343!
Making data for hr:9102.0 index: 345!
Making data for hr:9103.0 index: 355!
Making data for h

9030